In [11]:
import json
import os
from typing import Dict, List
import pandas as pd
import numpy as np
import time
import tushare as ts
import openai
import re

In [ ]:

class InformationBase:
    def __init__(self, filepath: str = "information_base.json", max_memory_size: int = 15):
        self.filepath = filepath
        self.max_memory_size = max_memory_size
        self.memory = self._load_or_init()

    def _load_or_init(self) -> Dict[str, List[str]]:
        """加载或初始化信息库文件"""
        if os.path.exists(self.filepath):
            with open(self.filepath, 'r', encoding='utf-8') as f:
                return json.load(f)
        else:
            initial_data = {
                "recommended_directions": ["尝试量价结合计算波动率。"],
                "prohibited_directions": ["禁止将绝对价格直接作为因子。"]
            }
            self._save(initial_data)
            return initial_data

    def _save(self, data: Dict = None):
        """持久化保存到 JSON"""
        data_to_save = data if data else self.memory
        with open(self.filepath, 'w', encoding='utf-8') as f:
            json.dump(data_to_save, f, ensure_ascii=False, indent=4)

    def update_experience(self, action: str, content: str):
        """
        更新记忆，并自动维护队列长度 (FIFO)
        action: 'add_recommended', 'add_prohibited', 'add_failed_attempt'
        """
        key_map = {
            "add_recommended": "recommended_directions",
            "add_prohibited": "prohibited_directions"
        }
        target_key = key_map.get(action)
        if target_key:
            self.memory[target_key].append(content)
            # 截断早期记忆，防止 LLM Context 爆炸
            self.memory[target_key] = self.memory[target_key][-self.max_memory_size:]
            self._save()

    def get_prompt_summary(self) -> str:
        """生成供 LLM 读取的字符串摘要"""
        return json.dumps(self.memory, ensure_ascii=False, indent=2)

In [3]:
class FactorBase:
    def __init__(self, metadata_path: str = "factor_base.json", data_dir: str = "./factor_data/"):
        self.metadata_path = metadata_path
        self.data_dir = data_dir
        os.makedirs(self.data_dir, exist_ok=True)
        self.factors_metadata = self._load_or_init()

    def _load_or_init(self) -> List[Dict]:
        if os.path.exists(self.metadata_path):
            with open(self.metadata_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        return[]

    def _save_metadata(self):
        with open(self.metadata_path, 'w', encoding='utf-8') as f:
            json.dump(self.factors_metadata, f, ensure_ascii=False, indent=4)

    def add_factor(self, formula: str, metrics: Dict, factor_df: pd.DataFrame):
        """入库新因子：保存元数据并落盘数据"""
        factor_id = f"alpha_{len(self.factors_metadata) + 1:04d}"
        
        # 1. 保存元数据
        new_record = {
            "factor_id": factor_id,
            "formula": formula,
            "metrics": metrics
        }
        self.factors_metadata.append(new_record)
        self._save_metadata()
        
        # 2. 保存计算好的 DataFrame 到本地 (使用 parquet 极速读写)
        file_path = os.path.join(self.data_dir, f"{factor_id}.parquet")
        factor_df.to_parquet(file_path)

    def get_all_factor_data(self) -> Dict[str, pd.DataFrame]:
        """加载所有已有因子的数据，用于正交化/相关性检验"""
        loaded_data = {}
        for factor in self.factors_metadata:
            fid = factor["factor_id"]
            file_path = os.path.join(self.data_dir, f"{fid}.parquet")
            if os.path.exists(file_path):
                loaded_data[fid] = pd.read_parquet(file_path)
        return loaded_data

    def get_prompt_summary(self) -> str:
        """只暴露公式给 LLM 避免过长"""
        formulas = [f["formula"] for f in self.factors_metadata]
        return "\n".join(formulas) if formulas else "暂无因子"

In [ ]:


class EvaluationEngine:
    def __init__(self, stock_data: Dict[str, pd.DataFrame], forward_returns: pd.DataFrame, factor_base: FactorBase):
        self.stock_data = stock_data          # 包含 'Close', 'Volume' 等
        self.forward_returns = forward_returns # T+1或T+5的收益率，用于计算IC
        self.factor_base = factor_base        # 注入因子库引用，用于算相关性
        
        # 注册安全的执行环境（将数据和算子注入）
        self.safe_env = self._register_operators()
        self.safe_env.update(self.stock_data)

    def _register_operators(self) -> dict:
        """
        全量化因子算子库 
        """
        # --- 辅助函数：安全算术 ---
        def safe_log(x): return np.log(x.abs().replace(0, np.nan))
        def safe_inv(x): return 1 / x.replace(0, np.nan)
        def safe_sqrt(x): return np.sqrt(x.abs())
        
        # --- 辅助函数：时序回归 (向量化计算提升速度) ---
        def roll_slope(x, d):
            # 使用 numpy 的 polyfit 提速，若含 NaN 则返回 NaN
            return x.rolling(window=d).apply(lambda v: np.polyfit(np.arange(d), v, 1)[0] if not np.isnan(v).any() else np.nan, raw=True)
            
        def roll_rsquare(x, d):
            return x.rolling(window=d).apply(lambda v: np.corrcoef(np.arange(d), v)[0, 1]**2 if not np.isnan(v).any() else np.nan, raw=True)
            
        def roll_resi(x, d):
            def _resi(v):
                if np.isnan(v).any(): return np.nan
                p = np.polyfit(np.arange(d), v, 1)
                return v[-1] - (p[0] * (d - 1) + p[1])
            return x.rolling(window=d).apply(_resi, raw=True)

        return {
            # 1. Arithmetic (算术运算 - 元素级)
            'Add': lambda x, y: x + y,
            'Sub': lambda x, y: x - y,
            'Mul': lambda x, y: x * y,
            'Div': lambda x, y: x / y.replace(0, np.nan),
            'Neg': lambda x: -x,
            'Abs': lambda x: x.abs(),
            'Sign': lambda x: np.sign(x),                 
            'Max2': lambda x, y: np.maximum(x, y),      
            'Min2': lambda x, y: np.minimum(x, y), 
            'Log': safe_log,
            'SignedPower': lambda x, e: np.sign(x) * (x.abs() ** e),
            'Power': lambda x, e: x ** e,
            'Inv': safe_inv,
            'Sqrt': safe_sqrt,
            'Square': lambda x: x ** 2,
            'Exp': lambda x: np.exp(x),
            'Tanh': lambda x: np.tanh(x),

            # 2. Statistical (统计特性 - 滚动窗口)
            'Mean': lambda x, d: x.rolling(window=d).mean(),
            'Std': lambda x, d: x.rolling(window=d).std(),
            'Var': lambda x, d: x.rolling(window=d).var(),
            'Skew': lambda x, d: x.rolling(window=d).skew(),
            'Kurt': lambda x, d: x.rolling(window=d).kurt(),
            'Med': lambda x, d: x.rolling(window=d).median(),
            'Sum': lambda x, d: x.rolling(window=d).sum(),
            'Product': lambda x, d: x.rolling(window=d).apply(np.prod, raw=True),

            # 3. Time-series (时序模式 - 滚动窗口)
            'Delay': lambda x, d: x.shift(d),
            'Delta': lambda x, d: x.diff(d),
            'TsRank': lambda x, d: x.rolling(window=d).rank(pct=True),
            'TsMax': lambda x, d: x.rolling(window=d).max(),
            'TsMin': lambda x, d: x.rolling(window=d).min(),
            'TsArgMax': lambda x, d: x.rolling(window=d).apply(np.argmax, raw=True),
            'TsArgMin': lambda x, d: x.rolling(window=d).apply(np.argmin, raw=True),
            'TsDecay': lambda x, d: x.rolling(window=d).apply(
                lambda v: np.dot(v, np.arange(1, d + 1)) / np.arange(1, d + 1).sum(), raw=True
            ), # 线性加权衰减
            'Cov': lambda x, y, d: x.rolling(window=d).cov(y),   
            'Corr': lambda x, y, d: x.rolling(window=d).corr(y), 
            
            # 4. Cross-sectional (截面变换)
            'CsRank': lambda x: x.rank(axis=1, pct=True),
            'Scale': lambda x, a=1: x.div(x.abs().sum(axis=1), axis=0) * a, # 标准化使绝对值和为a

            # 5. Smoothing (平滑/趋势提取)
            'SMA': lambda x, d: x.rolling(window=d).mean(),
            'EMA': lambda x, d: x.ewm(span=d, adjust=False).mean(),
            'WMA': lambda x, d: x.rolling(window=d).apply(
                lambda v: np.dot(v, np.arange(1, d + 1)) / np.arange(1, d + 1).sum(), raw=True
            ),

            # 6. Regression (回归提取 - 滚动时间窗口上的线性回归)
            'Slope': roll_slope,
            'Rsquare': roll_rsquare,
            'Resi': roll_resi,

            # 7. Logical (逻辑判断 - 状态切换)
            'IfElse': lambda cond, x, y: pd.DataFrame(np.where(cond, x, y), index=x.index, columns=x.columns),
            'Greater': lambda x, y: x > y,
            'Less': lambda x, y: x < y,
            'GreaterEqual': lambda x, y: x >= y,
            'LessEqual': lambda x, y: x <= y,
            'And': lambda x, y: x & y,
            'Or': lambda x, y: x | y,
            'Eq': lambda x, y: x == y,
            'Ne': lambda x, y: x != y
        }

    def _calculate_factor(self, formula: str) -> pd.DataFrame:
        """执行表达式树（生产环境应替换为 AST Parser）"""
        return eval(formula, {"__builtins__": {}}, self.safe_env)

    def evaluate(self, formula: str) -> tuple[bool, Dict, pd.DataFrame]:
        """
        核心流水线：传入公式，返回 (是否通过, 评测指标字典, 计算出的因子DataFrame)
        """
        try:
            # 1. 计算因子值
            factor_df = self._calculate_factor(formula)
            
            # 2. 评测 IC
            ic_series = factor_df.corrwith(self.forward_returns, axis=1, method='spearman')
            ic_mean = float(ic_series.mean())
            
            # 3. 评测换手率 (日均变化度)
            turnover = float(factor_df.diff().abs().mean(axis=1).mean())
            
            # 4. 评测红海相关性
            existing_factors = self.factor_base.get_all_factor_data()
            max_corr = 0.0
            for ext_df in existing_factors.values():
                corr = factor_df.corrwith(ext_df, axis=1).mean()
                if abs(corr) > max_corr:
                    max_corr = abs(corr)
                    
            metrics = {
                "IC": round(ic_mean, 4),
                "Turnover": round(turnover, 4),
                "Max_Correlation": round(max_corr, 4)
            }
            
            # 5. 综合判定条件
            is_success = (abs(ic_mean) > 0.03) and (turnover < 0.3) and (max_corr < 0.6)
            
            return is_success, metrics, factor_df

        except Exception as e:
            # 捕获公式异常（如未定义的变量、除零等）
            return False, {"Error": str(e)}, None

In [ ]:
class FactorMiningAgent:
    def __init__(self, info_base: InformationBase, factor_base: FactorBase, eval_engine: EvaluationEngine, api_key: str):
        self.info_base = info_base
        self.factor_base = factor_base
        self.eval_engine = eval_engine
        self.client = openai.Client(api_key=api_key,base_url="https://api.chatanywhere.tech/v1")

    def generate(self) -> str:
        """Step 1 & 2: 读取外部记忆，生成新公式"""
        prompt = f"""
        你是一个顶尖的量化因子挖掘专家。请根据信息库生成一个全新的 Alpha 因子公式。
        
        【可用数据列表】
        Close, Open, High, Low, Volume, Amount

        【可用算子库】
        1. 算术: Add(x,y), Sub(x,y), Mul(x,y), Div(x,y), Neg(x), Abs(x), Sign(x), Max2(x,y), Min2(x,y), Log(x), SignedPower(x,e), Power(x,e), Sqrt(x)
        2. 统计(滚动d天): Mean(x,d), Std(x,d), Skew(x,d), Kurt(x,d), Med(x,d), Sum(x,d)
        3. 时序与相关(滚动d天): Delay(x,d), Delta(x,d), TsRank(x,d), TsMax(x,d), TsMin(x,d), TsArgMax(x,d), TsDecay(x,d), Cov(x,y,d), Corr(x,y,d)
        4. 截面: CsRank(x), Scale(x)
        5. 平滑: SMA(x,d), EMA(x,d), WMA(x,d)
        6. 时序回归(滚动d天): Slope(x,d), Rsquare(x,d), Resi(x,d)
        7. 逻辑(返回条件判断): IfElse(cond, x, y), Greater(x,y), Less(x,y), And(x,y), Or(x,y), Eq(x,y)

        【公式示例】
        - 影线逻辑: Div(Sub(Min2(Open, Close), Low), Add(Sub(High, Low), 0.0001))
        - 量价协方差反转: Neg(Mul(TsRank(Cov(TsRank(Close,18), TsRank(Volume,18), 10), 18), CsRank(Returns)))
        - 状态切换: IfElse(Greater(Kurt(Returns, 24), 3), Neg(Resi(Close, 6)), Neg(Slope(Close, 12)))
        【已有因子】（禁止重复）:\n{self.factor_base.get_prompt_summary()}
        【信息库经验】（必须遵循）:\n{self.info_base.get_prompt_summary()}
        
        仅输出一行纯文本的数学公式。
        """
        response = self.client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            temperature=1
        )
        return response.choices[0].message.content.strip()

    def distill(self, formula: str, metrics: dict, is_success: bool):
        """Step 4: 基于评测结果进行反思，并更新信息库"""
        status = "成功" if is_success else "失败"
        prompt = f"""
        因子挖掘{status}。公式: {formula}。结果: {json.dumps(metrics)}
        请作为量化专家分析原因，并输出严格的 JSON 指令以更新经验库：
        {{
            "action": "add_recommended" 或 "add_prohibited" 
            "content": "具体的一句反思总结"
        }}
        注意：请绝对只输出这一个 JSON 字典！不要使用 markdown 语法(不要 ```json)，不要换行控制符，不要解释！反思的action严格参照是否成功！
        """
        try:
            response = self.client.chat.completions.create(
                model="deepseek-chat",  
                messages=[{"role": "user", "content": prompt}],
                temperature=1
            )
            raw_content = response.choices[0].message.content.strip()
            
            # 【清洗防御 1】：去除 Markdown 标记
            cleaned_content = raw_content.replace('```json', '').replace('```', '').strip()
            
            # 【清洗防御 2】：使用正则表达式强行提取大括号 {} 里的内容，防止 LLM 前后说废话
            match = re.search(r'\{.*\}', cleaned_content, re.DOTALL)
            if match:
                cleaned_content = match.group(0)
            
            # 【清洗防御 3】：去除可能导致 control character 报错的非法换行/制表符
            cleaned_content = cleaned_content.replace('\n', ' ').replace('\r', '').replace('\t', ' ')
            
            # 尝试解析
            instruction = json.loads(cleaned_content)
            
            action = instruction.get("action")
            content = instruction.get("content")
            
            if action and content:
                self.info_base.update_experience(action, content)
                # print(f"  [反思记录] {action}: {content}") # 如果需要看反思内容可以取消注释
                
        except json.JSONDecodeError as e:
            # 如果解析依然失败，打印错误，但使用 pass 跳过，保证循环继续
            print(f"\n⚠️ [解析跳过] LLM 输出格式异常无法解析。原始输出: {raw_content}")
        except Exception as e:
            # 捕获其他任何网络断开或 API 错误
            print(f"\n⚠️ [网络/API 异常] 反思阶段出错: {e}")

    def run_ralph_loop(self, max_iterations: int = 100):
        """核心无状态迭代循环"""
        for i in range(max_iterations):
            print(f"\n--- 启动第 {i+1} 轮迭代 ---")
            
            # 1. 生成因子
            formula = self.generate()
            print(f"Agent 生成公式: {formula}")
            
            # 2. 客观评测
            is_success, metrics, factor_df = self.eval_engine.evaluate(formula)
            print(f"评测结果: 是否成功={is_success}, 指标={metrics}")
            
            # 3. 提炼反思
            self.distill(formula, metrics, is_success)
            
            # 4. 入库结算
            if is_success and factor_df is not None:
                self.factor_base.add_factor(formula, metrics, factor_df)
                print(f"✅ 因子入库成功！ID 记录已更新。")
                
            time.sleep(1) # 避免 API 限流

In [6]:
class TushareDataLoader:
    def __init__(self, token: str, start_date: str, end_date: str,stock_id=[], cache_dir: str = "./data_cache/"):
        self.token = token
        self.start_date = start_date
        self.end_date = end_date
        self.cache_dir = cache_dir
        self.stockid= stock_id

        ts.set_token(self.token)
        self.pro = ts.pro_api()
        os.makedirs(self.cache_dir, exist_ok=True)

    def fetch_daily_data(self) -> dict:
        """按股票列表循环获取日线数据，并转换为宽表字典 (使用 PyArrow 引擎)"""
        # 使用量化标准的 .parquet 格式缓存
        cache_file = os.path.join(self.cache_dir, f"all_a_shares_{self.start_date}_{self.end_date}.parquet")
        
        # 1. 尝试从本地缓存读取
        if os.path.exists(cache_file):
            print(f"命中本地缓存: {cache_file}，正在使用 PyArrow 飞速加载全市场数据...")
            all_data = pd.read_parquet(cache_file)
        else:
            # ==========================================
            # 第一步：顺延日期，直到找到有效交易日并获取全市场股票列表
            # ==========================================
            current_date = pd.to_datetime(self.start_date)
            end_dt = pd.to_datetime(self.end_date)
            stock_list =[]
            
            print(f"正在寻找 {self.start_date} 之后的首个有效交易日...")
            while current_date <= end_dt:
                date_str = current_date.strftime('%Y%m%d')
                try:
                    # 尝试拉取这一天的数据
                    df_day = self.pro.daily(trade_date=date_str)
                    if df_day is not None and not df_day.empty:
                        # 如果非空，说明是交易日，提取股票代码列表
                        stock_list = df_day['ts_code'].tolist()
                        print(f"✅ 成功找到首个交易日 {date_str}，共获取到 {len(stock_list)} 只股票代码！")
                        break
                    else:
                        print(f"  [-] {date_str} 为非交易日，顺延下一天...")
                except Exception as e:
                    print(f"  [x] 请求 {date_str} 失败: {e}")
                
                # 往后加一天
                current_date += pd.Timedelta(days=1)
                time.sleep(0.3)
                
            if not stock_list:
                raise ValueError(f"在 {self.start_date} 到 {self.end_date} 期间，未找到任何有效的交易日数据！")

            # ==========================================
            # 第二步：对股票列表进行循环，分别提取该股票区间数据
            # ==========================================
            if self.stockid:
                all_data_list =[]
                total_stocks = len(self.stockid)
                stock_list=self.stockid
            else:
                all_data_list =[]
                total_stocks = len(stock_list)
            print(f"\n开始按股票代码循环提取数据，总计需调用 API {total_stocks} 次...")
            print("⚠️ 注意：按股票循环请求预计需要较长时间，请保持网络畅通并耐心等待...")
            for i, code in enumerate(stock_list):
                try:
                    df = self.pro.daily(ts_code=code, start_date=self.start_date, end_date=self.end_date)
                    if df is not None and not df.empty:
                        all_data_list.append(df)
                    # 严格控制休眠防封，0.15 秒是极限边缘，如果依然被封请加到 0.2 或 0.3
                    time.sleep(1.2) 
                    
                    # 每完成 100 只打印一次进度
                    if (i + 1) % 100 == 0:
                        print(f"进度：已提取 {i + 1} / {total_stocks} 只股票...")
                        
                except Exception as e:
                    print(f"下载 {code} 截面数据失败: {e}")
                    time.sleep(1) # 遇到网络波动稍微多等一会
                    
            # ==========================================
            # 第三步：拼接数据与缓存落盘 (Parquet)
            # ==========================================
            print("\n正在拼接全市场数据表...")
            all_data = pd.concat(all_data_list, ignore_index=True)
            
            # 格式化日期并排序
            all_data['trade_date'] = pd.to_datetime(all_data['trade_date'])
            all_data.sort_values(['trade_date', 'ts_code'], inplace=True)
            
            # 使用 PyArrow 引擎保存到本地 parquet 缓存
            all_data.to_parquet(cache_file, engine='pyarrow')
            print("全股票数据下载完成并已安全落盘 (Parquet格式)！\n")

        # ==========================================
        # 第四步：数据透视（长表转宽表矩阵）
        # ==========================================
        print("正在构建量化计算引擎数据矩阵 (长表转宽表)...")
        stock_data = {}
        features =['open', 'high', 'low', 'close', 'vol', 'amount']
        
        for feature in features:
            # 行是日期，列是这 5000 多只股票的代码
            pivot_df = all_data.pivot(index='trade_date', columns='ts_code', values=feature)
            pivot_df.sort_index(inplace=True)
            
            # 全市场数据会存在很多停牌/未上市导致的 NaN，执行向下填充处理防止因子计算断裂
            pivot_df = pivot_df.ffill() 
            
            key_name = 'Volume' if feature == 'vol' else feature.capitalize()
            stock_data[key_name] = pivot_df

        return stock_data

    def build_forward_returns(self, close_df: pd.DataFrame, periods: int = 1) -> pd.DataFrame:
        """构建未来收益率矩阵 (用于 IC 测试)"""
        forward_returns = close_df.shift(-periods) / close_df - 1.0
        return forward_returns

In [ ]:
if __name__ == "__main__":
    # 配置你的 Token 和 API Key
    TUSHARE_TOKEN = "1067ed1160b1dedeca68f122160aca415393dfd50f9f49ccbd1d0580"   # 去 Tushare 官网获取
    OPENAI_API_KEY = "sk-CqqsrgnNF94QRUDkpWmuC8NvVU12ptKDGycqnGzUZUz5Jy75"  # 大模型 API Key

    print("\n[System] 启动全市场因子挖掘环境...")
    
    # 1. 实例化 Tushare 数据加载器 (修改为 2025年6月1日 到 2025年9月30日)
    data_loader = TushareDataLoader(
        token=TUSHARE_TOKEN, 
        start_date="20240601", 
        end_date="20250930",
        stock_id=["000001.sz","000002.sz","000003.sz","000004.sz","000005.sz","000006.sz","000007.sz","000008.sz"]
    )
    
    # 获取全市场 5000+ 股票的宽表矩阵
    stock_data = data_loader.fetch_daily_data()
    
    # 收益率矩阵构建
    forward_returns = data_loader.build_forward_returns(stock_data['Close'], periods=1)

    print("\n[System] 初始化智能体依赖库...")
    
    # 2. 实例化记忆存储 (区分存储，避免与之前沪深300的混淆)
    info_memory = InformationBase("all_market_info_base.json")
    factor_assets = FactorBase("all_market_factor_base.json", "./all_market_factor_data/")

    # 3. 实例化评测引擎
    engine = EvaluationEngine(
        stock_data=stock_data, 
        forward_returns=forward_returns, 
        factor_base=factor_assets
    )

    # 4. 实例化因子挖掘 Agent 
    agent = FactorMiningAgent(
        info_base=info_memory, 
        factor_base=factor_assets, 
        eval_engine=engine, 
        api_key=OPENAI_API_KEY
    )

    print("\n[System] 数据加载完毕，开始在【全市场环境】执行 Ralph Loop 智能迭代因子挖掘...")
    
    # 5. 启动迭代
    try:
        agent.run_ralph_loop(max_iterations=2)
    except KeyboardInterrupt:
        print("\n[System] 收到中断信号，因子挖掘系统已安全停止。")


[System] 启动全市场因子挖掘环境...
命中本地缓存: ./data_cache/all_a_shares_20240601_20250930.parquet，正在使用 PyArrow 飞速加载全市场数据...
正在构建量化计算引擎数据矩阵 (长表转宽表)...

[System] 初始化智能体依赖库...

[System] 数据加载完毕，开始在【全市场环境】执行 Ralph Loop 智能迭代因子挖掘...

--- 启动第 1 轮迭代 ---
Agent 生成公式: Ts_Max(Rank(Delay(Delta(Close, 5), 2) / Ts_Mean(Volume, 20)), 10) - Abs(Correlation(Volume, Close, 10))
评测结果: 是否成功=True, 指标={'IC': 0.0307, 'Turnover': 0.1322, 'Max_Correlation': 0.0}
✅ 因子入库成功！ID 记录已更新。

--- 启动第 2 轮迭代 ---
Agent 生成公式: Rank(Ts_Max(Delta(Close, 3) * Volume / Ts_Mean(Abs(Delta(Close, 1)), 10), 5))
评测结果: 是否成功=True, 指标={'IC': -0.037, 'Turnover': 0.0872, 'Max_Correlation': np.float64(0.1362)}
✅ 因子入库成功！ID 记录已更新。

--- 启动第 3 轮迭代 ---
Agent 生成公式: Rank(Ts_Mean(Abs(Delta(Close, 3)) * Volume / Ts_Mean(Volume, 20), 10))
评测结果: 是否成功=True, 指标={'IC': -0.0412, 'Turnover': 0.0492, 'Max_Correlation': np.float64(0.1886)}
✅ 因子入库成功！ID 记录已更新。

--- 启动第 4 轮迭代 ---
Agent 生成公式: Rank(Correlation(Abs(Delta(Close, 2)), Ts_Mean(Volume, 10), 5) / Delta(Ts_Mean(Abs(De

RateLimitError: Error code: 429 - {'error': {'message': 'deepseek-chat模型免费API限制每日30次请求，请00:00后再试，如有更多需求，请访问 https://api.chatanywhere.tech/#/shop 购买付费API。The free account is limited to 30 requests per day. Please try again after 00:00 the next day. If you have additional requirements, please visit https://api.chatanywhere.tech/#/shop to purchase a premium key.(当前请求使用的ApiKey: sk-Cqq****Jy75)【如果您遇到问题，欢迎加入QQ群咨询：836739524】', 'type': 'chatanywhere_error', 'param': None, 'code': '429 TOO_MANY_REQUESTS'}}

In [ ]:
"""
可以优化的方向：
1.用更好的大语言模型
2.采用更多的大模型，分属不同职位
3.多模态输入
4.增添算子和机器学习算子
5.增添因子评价
6.优化prompt结构，采用cot
7.输入数据细粒度



"""

{'Open': ts_code     000001.SZ  000002.SZ  000004.SZ  000006.SZ  000007.SZ  000008.SZ  \
 trade_date                                                                     
 2025-09-29        NaN        NaN        NaN        NaN        NaN        NaN   
 2025-09-30      11.37        6.8      11.12       9.48        7.2       2.88   
 
 ts_code     000009.SZ  000010.SZ  000011.SZ  000012.SZ  ...  920964.BJ  \
 trade_date                                              ...              
 2025-09-29        NaN        NaN        NaN        NaN  ...       7.35   
 2025-09-30      12.62        4.3       9.01       4.68  ...       7.49   
 
 ts_code     920970.BJ  920971.BJ  920974.BJ  920976.BJ  920978.BJ  920981.BJ  \
 trade_date                                                                     
 2025-09-29       9.58      38.49       8.44      25.36      41.45      36.36   
 2025-09-30       9.60      39.95       8.65      26.31      40.30      37.36   
 
 ts_code     920982.BJ  920985.BJ  920